# CPT: Continued Pre-Training of Llama-3.2-1B on Automotive E/E Data Corpus from part1

- Adapts the base Llama-3.2-1B model to automotive E/E text via continued pre-training on the part1 data corpus.
- Uses LoRA and data mixing with WikiText to limit catastrophic forgetting.
- Best checkpoint selected by validation loss (with held-out E/E validation set).
- Runs on Kaggle T4: Toy setup with a few minutes of training.

## S0: Configuration

In [ ]:
GITHUB_REPO    = "https://github.com/tillacs/Domain-Adapted-LLM-Training-for-Automotive-E-E.git"
MODEL_NAME     = "unsloth/Llama-3.2-1B-bnb-4bit"
OUTPUT         = "/kaggle/working/cpt_merged"

MAX_SEQ_LENGTH =  2048 # Same as Chunk Size

LORA_RANK        = 64
LORA_ALPHA       = 64
LR    = 1e-4
LR_EMB= 1e-5
TARGET_MODULES   = ["q_proj", "k_proj", "v_proj", "o_proj",
                                  "gate_proj", "up_proj", "down_proj",
                                  "embed_tokens", "lm_head"]
NUM_EPOCHS       = 2
SEED             = 42

## S1: Install unsloth


In [ ]:
!pip install unsloth unsloth_zoo bitsandbytes

## S2: Load compressed Model


In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

## S3: Part1 Training Data + Data Mixing



In [ ]:
import json
import random
import math
from datasets import Dataset, load_dataset

!git clone -q {GITHUB_REPO} repo


# 3a. Decode token IDs from Part1 back to text, append EOS

def load_texts(path):
    with open(path) as f:
        ids = json.load(f)
    return [t + tokenizer.eos_token for t in tokenizer.batch_decode(ids, skip_special_tokens=True)]

train_texts = load_texts("repo/part1_data/data_corpus/train.json")
val_dataset = Dataset.from_dict({"text": load_texts("repo/part1_data/data_corpus/val.json")})


# 3b. Data Mixing with WikiText

wiki   = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
n_wiki = max(1, int(len(train_texts) * 0.05))
wiki_texts = [t + tokenizer.eos_token for t in wiki["text"] if len(t.strip()) > 100][:n_wiki]

random.seed(SEED)
all_texts = train_texts + wiki_texts
random.shuffle(all_texts)

dataset = Dataset.from_dict({"text": all_texts}) # build HugginFace dataset-object

print(f"Domain: {len(train_texts)} | General: {len(wiki_texts)} | Total: {len(dataset)} | Train: {len(train_texts)} | Val: {len(val_dataset)}")

## S4: Pre-CPT Inference



In [ ]:
TEST_PROMPT = ["The AUTOSAR software architecture consists of"]

@torch.no_grad()
def perplexity():
    loss, n = 0.0, 0
    for ex in val_dataset:
        ids = tokenizer(ex["text"], return_tensors="pt", truncation=True,
                        max_length=MAX_SEQ_LENGTH).input_ids.to("cuda")
        loss += model(ids, labels=ids).loss.item() * ids.shape[1]; n += ids.shape[1]
    return math.exp(loss / n)

@torch.no_grad()
def show_samples():
        out = model.generate(**tokenizer(TEST_PROMPT, return_tensors="pt").to("cuda"),
                             max_new_tokens=150, max_length=None, do_sample=False)  # greedy
        print(f"\n{TEST_PROMPT}\n{tokenizer.decode(out[0], skip_special_tokens=True)}")
    

FastLanguageModel.for_inference(model) # Switch to inference mode
model.eval()
print(f"Pre-CPT PPL: {perplexity():.2f}")
show_samples()
FastLanguageModel.for_training(model) # Switch back to training mode before LoRA setup and training

## S5: LoRA Setup


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_RANK,
    lora_alpha                 = LORA_ALPHA,
    target_modules             = TARGET_MODULES,
    random_state               = SEED,
    use_rslora                 = True, # scaling α/√r → stable across different r
    use_gradient_checkpointing = "unsloth",
)

## S6: Training

In [ ]:
from unsloth import UnslothTrainer, UnslothTrainingArguments, is_bfloat16_supported

trainer = UnslothTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    eval_dataset       = val_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    args = UnslothTrainingArguments(
        output_dir                  = "/kaggle/working/cpt_output",
        num_train_epochs            = NUM_EPOCHS,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        learning_rate               = LR,
        embedding_learning_rate     = LR_EMB,
        lr_scheduler_type           = "cosine",
        warmup_steps                = 6,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        optim                       = "adamw_8bit",
        logging_steps               = 1,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,  # keep the best val-loss checkpoint
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        seed                        = SEED,
    ),
)

stats = trainer.train()

print(f"Post-CPT PPL: {math.exp(trainer.state.best_metric):.2f} | {stats.metrics['train_runtime']/60:.1f} min")

## S7: Loss Curve

In [ ]:
import matplotlib.pyplot as plt
losses = [x["loss"] for x in trainer.state.log_history if "loss" in x]
plt.figure(figsize=(9, 3))
plt.plot(losses, lw=1.2); plt.xlabel("Step"); plt.ylabel("Loss")
plt.title("CPT Loss — Llama-3.2-1B, r=64"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## S8: Post CPT Inference

In [ ]:
print("── Post-CPT samples ──")
FastLanguageModel.for_inference(model) # Switch to inference mode
model.eval()
show_samples()

## S9: Save Model

In [ ]:
# merged 16-bit model to Kaggle-Output
model.save_pretrained_merged(OUTPUT, tokenizer, save_method="merged_16bit")
print("Saved merged 16-bit CPT model to:", OUTPUT)